# Phase 02 — Penggabungan Bank, Pemecahan Kalimat & EDA

Notebook **mandiri (self-contained)**. Mengubah data mentah artikel (hasil Phase 01) menjadi data **per kalimat** untuk BERTopic (Phase 03), lalu melakukan **EDA** sebagai **analisis saja**.

### Prinsip
- **Dateline** ('Kota, CNBC Indonesia -') sudah dibuang di Phase 01, jadi tidak ditangani lagi di sini.
- **Bagian EDA bersifat ANALISIS, bukan transformasi.** Tidak ada perubahan permanen (casefolding / stopword removal / drop) pada kolom `kalimat` yang disimpan — teks tetap asli & alami agar embedding BERTopic tidak rusak. Casefolding & stopword hanya *ditampilkan* untuk memahami data. Kesimpulan & keputusan dirangkum di sel akhir.
- Satu-satunya pengurangan data: buang kalimat < 5 kata (filter kualitas) + duplikat + kosong (noise sah).
- Kalimat TIDAK difilter berdasarkan kata relevan (aspek harus induktif dari BERTopic).

### Alur
1. Baca `data/raw/raw.csv`
2. Gabung keyword → kolom `bank`
3. Cleaning ringan (footer/email/label — BUKAN casefolding/stopword)
4. Pecah per kalimat (lindungi singkatan `PT.`/`Tbk.`/desimal)
5. **EDA (analisis saja)**: wordcount, panjang kalimat, duplikat, missing, frekuensi kata, stopword, casefolding preview
6. Buang kalimat < 5 kata + duplikat + kosong
7. Kesimpulan
8. Simpan `data/processed/sentences.csv`

## 1. Import & baca data

In [51]:
import re
from pathlib import Path
from collections import Counter
import pandas as pd

In [52]:
df = pd.read_csv("data/raw/raw.csv", dtype={"id_artikel": str})
print(f"Artikel mentah: {len(df)}")
df.head(3)

Artikel mentah: 3979


,id_artikel,keyword,url,timestamp,judul,isi_mentah
0,777827,BCA,https://www.cnbcindonesia.com/market/202610201...,2026-10-20,"Aksi Jual Asing Rp1,97 T di Sesi I, Saham BCA ...",Indeks Harga Saham Gabungan (IHSG) berhasil me...
1,777666,BCA,https://www.cnbcindonesia.com/market/202610192...,2026-10-19,"RI Alami Halusinasi Daya Beli, Ekonom Ungkap F...",Kepala Ekonom PT Bank Central Asia Tbk David E...
2,777293,BCA,https://www.cnbcindonesia.com/market/202610190...,2026-10-19,"BCA Mau Buyback 19-21 Oktober, Karyawan Dilara...",PT Bank Central Asia Tbk (BBCA) menginformasik...


## 2. Gabung keyword → kolom `bank`

Keyword scraping (BCA, BBCA, BRI, ...) digabung jadi 4 bank. Deteksi dari **judul + isi** (satu artikel sering menyebut beberapa bank). Kolom `bank` berisi daftar bank yang disebut, dipisah koma.

In [54]:
POLA_BANK = {
    "BCA":     r"\bbca\b|\bbbca\b|bank central asia", 
    "BRI":     r"\bbri\b|\bbbri\b|bank rakyat indonesia|brilink",
    "BNI":     r"\bbni\b|\bbbni\b|bank negara indonesia",
    "Mandiri": r"\bmandiri\b|\bbmri\b|bank mandiri",
}

def deteksi_bank(teks):
    teks = (teks or "").lower()
    return ", ".join(b for b, pat in POLA_BANK.items() if re.search(pat, teks))

df["bank"] = (df["judul"].fillna("") + " " + df["isi_mentah"].fillna("")).map(deteksi_bank)
print("Distribusi jumlah bank yang disebut per artikel:")
print(df["bank"].map(lambda x: len(x.split(", ")) if x else 0).value_counts().sort_index())
print("Artikel tanpa bank :", (df["bank"] == "").sum())
df[["id_artikel", "keyword", "bank", "judul"]].head(5)

Distribusi jumlah bank yang disebut per artikel:
bank
0       4
1    1868
2     754
3     908
4     445
Name: count, dtype: int64
Artikel tanpa bank : 4


,id_artikel,keyword,bank,judul
0,777827,BCA,"BCA, BRI, Mandiri","Aksi Jual Asing Rp1,97 T di Sesi I, Saham BCA ..."
1,777666,BCA,BCA,"RI Alami Halusinasi Daya Beli, Ekonom Ungkap F..."
2,777293,BCA,BCA,"BCA Mau Buyback 19-21 Oktober, Karyawan Dilara..."
3,776824,BCA,"BCA, BRI, BNI, Mandiri","Asing Keluar Rp2,92 Triliun di Sesi I, Saham B..."
4,776759,BCA,"BCA, BRI, BNI, Mandiri",Ada Bank Yang Sudah Jual Dolar di Rp18.300


## 3. Cleaning ringan

Buang noise struktural (footer research, email, label 'Baca:'/'Pilihan Redaksi') + normalisasi spasi. **Tidak** casefolding, **tidak** stopword removal, **tidak** menyaring berdasarkan topik.

In [ ]:
def cleaning_ringan(teks):
    t = str(teks)
    t = re.sub(r"CNBC INDONESIA RESEARCH\s*\S*@\S*", " ", t, flags=re.I)   # footer research
    t = re.sub(r"\S+@\S+\.\S+", " ", t)                                    # email
    t = re.sub(r"Baca\s*:\s*", " ", t, flags=re.I)                         # label 'Baca:'
    t = re.sub(r"Pilihan Redaksi", " ", t, flags=re.I)
    t = re.sub(r"\s+", " ", t).strip()
    return t

df["isi_bersih"] = df["isi_mentah"].map(cleaning_ringan)

# Bandingkan sebelum vs sesudah cleaning
for i in range(3):   # <- bisa diubah: jumlah contoh yang dibandingkan
    print(f"===== Contoh {i+1} (id_artikel: {df['id_artikel'].iloc[i]}) =====")
    print("SEBELUM:", df["isi_mentah"].iloc[i][:200])
    print("SESUDAH:", df["isi_bersih"].iloc[i][:200])
    print()

===== Contoh 1 (id_artikel: 777827) =====
SEBELUM: Indeks Harga Saham Gabungan (IHSG) berhasil mengakhiri perdagangan sesi I hari ini, Selasa (20/10/2026), di zona hijau. IHSG ditutup di level 5.204,63, naik 208,02 poin atau 4,16%. Namun, asing masih 
SESUDAH: Indeks Harga Saham Gabungan (IHSG) berhasil mengakhiri perdagangan sesi I hari ini, Selasa (20/10/2026), di zona hijau. IHSG ditutup di level 5.204,63, naik 208,02 poin atau 4,16%. Namun, asing masih 

===== Contoh 2 (id_artikel: 777666) =====
SEBELUM: Kepala Ekonom PT Bank Central Asia Tbk David E. Sumual, mengatakan konsumsi masyarakat Indonesia tengah mengalami fenomena lipstick effect. Lipstick effect adalah fenomena psikologis dan ekonomi masya
SESUDAH: Kepala Ekonom PT Bank Central Asia Tbk David E. Sumual, mengatakan konsumsi masyarakat Indonesia tengah mengalami fenomena lipstick effect. Lipstick effect adalah fenomena psikologis dan ekonomi masya

===== Contoh 3 (id_artikel: 777293) =====
SEBELUM: PT Bank Central Asia Tb

## 4. Pemecah kalimat (melindungi singkatan Indonesia)

Pemecah naif memotong salah di `PT.`, `Tbk.`, angka desimal (`1.519`). Fungsi ini melindungi titik singkatan dulu, baru memecah di `. ! ?`. Hasil: tabel kalimat dengan `id_artikel`, `id_kalimat`, `bank`, `timestamp`.

In [6]:
SINGKATAN = ["PT", "Tbk", "Persero", "Rp", "US", "No", "Jl", "Dr", "Ir", "Hj",
             "dll", "dsb", "dkk", "a.l", "yakni"]   # <- bisa diubah: tambah singkatan bila perlu

def pecah_kalimat(teks):
    t = re.sub(r"\s+", " ", str(teks)).strip()
    for ab in SINGKATAN:
        t = re.sub(rf"\b{re.escape(ab)}\.", f"{ab}<DOT>", t)
    t = re.sub(r"(?<=\d)\.(?=\d)", "<DOT>", t)
    kalimat = re.split(r"(?<=[.!?])\s+", t)
    return [k.replace("<DOT>", ".").strip() for k in kalimat if k.strip()]

baris = []
for _, r in df.iterrows():
    for i, k in enumerate(pecah_kalimat(r["isi_bersih"])):
        baris.append({
            "id_artikel": r["id_artikel"],
            "id_kalimat": f"{r['id_artikel']}_{i}",
            "bank": r["bank"],
            "timestamp": r["timestamp"],
            "kalimat": k,
        })
kal = pd.DataFrame(baris)
kal["n_kata"] = kal["kalimat"].str.split().str.len()
print(f"Total kalimat mentah: {len(kal)} dari {len(df)} artikel")
print(f"Rata-rata kalimat/artikel: {len(kal)/len(df):.1f}")

Total kalimat mentah: 82895 dari 3979 artikel
Rata-rata kalimat/artikel: 20.8


## 5. EDA — ANALISIS SAJA (tidak mengubah data)

Semua sel di bawah hanya **memeriksa** data; tidak ada perubahan permanen. Kesimpulan dirangkum di bagian 7.

### 5a. Distribusi wordcount & panjang kalimat

In [8]:
print("Statistik jumlah kata per kalimat:")
print(kal["n_kata"].describe().round(1))

Statistik jumlah kata per kalimat:
count    82895.0
mean        19.2
std         13.9
min          1.0
25%         12.0
50%         17.0
75%         23.0
max        437.0
Name: n_kata, dtype: float64


In [56]:
print("Distribusi panjang kalimat (kata):")
bins = [0, 5, 10, 15, 20, 30, 50, 10000]
print(pd.cut(kal["n_kata"], bins=bins, right=False).value_counts().sort_index())

Distribusi panjang kalimat (kata):
n_kata
[0, 5)          1329
[5, 10)         8841
[10, 15)       21621
[15, 20)       20613
[20, 30)       21431
[30, 50)        7465
[50, 10000)     1595
Name: count, dtype: int64


### 5b. Duplikat & missing (hanya dihitung, belum dibuang)

**Duplicate**

In [12]:
kal["kalimat"].duplicated().sum()

np.int64(17373)

In [16]:
kal[kal["kalimat"].duplicated() == True]

,id_artikel,id_kalimat,bank,timestamp,kalimat,n_kata
121,776155,776155_2,BCA,2026-10-14,"Sementara itu, untuk tenor menengah, seperti F...",19
122,776155,776155_3,BCA,2026-10-14,Sedangkan tenor panajang seperti FR0107 hanya ...,13
123,776155,776155_4,BCA,2026-10-14,"Dibanding tahun lalu, tepatnya hasil lelang pa...",27
124,776155,776155_5,BCA,2026-10-14,Sedangkan untuk tenor menengah seperti FR0109 ...,17
125,776155,776155_6,BCA,2026-10-14,Adapun untuk tenor panjang saat itu seperti FR...,23
...,...,...,...,...,...,...
82863,608753,608753_18,Mandiri,2025-02-07,Meskipun Kanada dan Meksiko mengalami penundaa...,10
82864,608753,608753_19,Mandiri,2025-02-07,Negeri Tirai Bambu kemudian membalas tarif imp...,8
82877,608753,608753_32,Mandiri,2025-02-07,CNBC INDONESIA RESEARCH,3
82884,608414,608414_6,Mandiri,2025-02-06,"Ditambah, dalam waktu dekat ini juga akan ada ...",19


In [23]:
print(kal.loc[kal["id_kalimat"] == "776155_2", "kalimat"].iloc[0])

Sementara itu, untuk tenor menengah, seperti FR0110 bid to cover ratio nya 1,94 kali dan FR0111 sebanyak 2,08 kali.


In [ ]:
# Jumlah kalimat yang paling banyak terduplicate
dup_counts = kal["kalimat"].value_counts()
for s, c in dup_counts[dup_counts > 1].head(30).items():
    print(f"  ({c}x) {s[:80]}")

  (127x) Keputusan berinvestasi sepenuhnya berada di tangan masing-masing investor sesuai
  (127x) Selamat berinvestasi secara bijak.
  (113x) 1.
  (76x) 2.
  (61x) 3.
  (54x) 4.
  (44x) 5.
  (34x) Keputusan sepenuhnya ada pada diri pembaca, sehingga kami tidak bertanggung jawa
  (32x) Analisis ini tidak bertujuan mengajak pembaca untuk membeli, menahan, atau menju
  (27x) Nasabah BRI Prioritas bisa mendapatkan BRI Prioritas Debit Card yang menyediakan
  (26x) Sanggahan: Artikel ini adalah produk jurnalistik berupa pandangan CNBC Indonesia
  (26x) 6.
  (23x) Mengutip Stockbit, berikut net foreign sell perdagangan Rabu!
  (18x) Mengutip Stockbit, berikut net foreign sell perdagangan Senin!
  (18x) Kemudian, manfaat spesial bagi nasabah Mandiri Prioritas adalah 4x Livin'poin un
  (18x) Selain itu, ada perlindungan dari pembelian barang, perlindungan asuransi perjal
  (18x) Ltd.
  (18x) "Dalam sekejap, seluruh mobil di garasi terbakar, termasuk juga seisi rumah.
  (18x) Nasabah istimewa B

**Missing Text**

In [26]:
(kal["kalimat"].str.strip() == "").sum()

np.int64(0)

**Kalimat dibawah 5**

In [30]:
n_pendek = (kal["n_kata"] < 5).sum()
print(f"Kalimat < 5 kata : {n_pendek} ({n_pendek/len(kal)*100:.1f}%)")

Kalimat < 5 kata : 1329 (1.6%)


In [38]:
pendek = kal[kal["n_kata"] == 5]
print(f"Distribusi jumlah kata pada kalimat pendek: {dict(pendek['n_kata'].value_counts().sort_index())}")

Distribusi jumlah kata pada kalimat pendek: {5: np.int64(547)}


In [39]:
print("\n30 contoh kalimat < 5 kata (pastikan memang sampah, bukan relevan):")
for s in pendek["kalimat"].head(50).tolist():
    print("  |", s)


30 contoh kalimat < 5 kata (pastikan memang sampah, bukan relevan):
  | Haryn dalam keterangannya, Jumat (25/9/2026).
  | Prediksi itu kemudian benar terjadi.
  | Haryn dalam keterangannya, Jumat (25/9/2026).
  | Haryn dalam keterangannya, Jumat (25/9/2026).
  | Haryn dalam keterangannya, Jumat (25/9/2026).
  | Pengalaman serupa dibagikan akun @mawarniseptiana.
  | laporannya ke official IG Dana.
  | Hati-hati akun palsu ya!" tulisnya.
  | Haryn dalam keterangannya, Jumat (25/9/2026).
  | Salah satunya, Mall Taman Anggrek.
  | "Yang pasti rekam jejak beliau.
  | Belakangan, keyakinan itu seolah terbukti.
  | Belakangan, keyakinan itu seolah terbukti.
  | Prediksi itu kemudian benar terjadi.
  | Namun, pembacaan itu perlu hati-hati.
  | Prediksi itu kemudian benar terjadi.
  | Ada perhitungan presisi dan khusus.
  | Berikut daftarnya mengutip Indo Premier!
  | Ada perhitungan presisi dan khusus.
  | Prediksi itu kemudian benar terjadi.
  | Harap segera tukarkan poin Anda.
  | "Pinjaman

Ada beberapa kata dibawah 5, yang memang bermakna ("BCA menjadi yang terparah."). dan untuk kata - kata lain yang hanya 5 kata saja bermakna, sehingga filternya hanya akan menghapus data < 4

### 5d. Relevansi kata: frekuensi kata (analisis, bukan untuk dibuang)

Catatan: kata ber-frekuensi tinggi di sini **bukan noise untuk dibuang** — sebagian stopword (yang/dan/di) ditangani nanti di vectorizer BERTopic (Phase 03), sebagian lagi kata domain (rp/bank/saham) yang justru inti topik. Tabel ini hanya untuk memahami sebaran kata.

In [40]:
tokens = []
for k in kal["kalimat"]:
    tokens += re.findall(r"[a-zA-Z]+", str(k).lower())
freq = Counter(tokens)
print(f"Total token: {len(tokens)} | kata unik: {len(freq)}")
print("\n30 kata paling sering:")
for w, c in freq.most_common(30):
    print(f"  {w:18} {c}")

Total token: 1505477 | kata unik: 21107

30 kata paling sering:
  dan                30132
  rp                 29739
  yang               28678
  di                 23068
  saham              22419
  pt                 18538
  miliar             18036
  bank               17181
  tbk                16717
  pada               16612
  dengan             16387
  ini                16057
  dari               13586
  indonesia          13434
  dalam              11258
  menjadi            10169
  pasar              8892
  untuk              8833
  triliun            8730
  ihsg               8090
  itu                8065
  indeks             8029
  ke                 7796
  juga               7483
  sebesar            7407
  perdagangan        7188
  akan               6897
  atau               6383
  tersebut           6354
  hari               6134


### 5e. Analisis stopword & casefolding (PREVIEW saja — tidak diterapkan ke data)

Menampilkan seberapa besar porsi stopword dan bentuk kalimat bila di-casefold. Ini hanya gambaran; kolom `kalimat` yang disimpan tetap ASLI.

In [50]:
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
stopwords = set(StopWordRemoverFactory().get_stop_words())

n_stop = sum(1 for t in tokens if t in stopwords)

contoh = kal["kalimat"].iloc[42]
print("asli        :", contoh[:90])
print("casefold    :", contoh.lower()[:90])

tanpa_stop = " ".join(w for w in re.findall(r"\w+", contoh.lower()) if w not in stopwords)
print("tanpa stop  :", tanpa_stop[:90])

asli        : Termasuk terkait dengan transaksi yang kini makin banyak beralih melalui ruang digital.
casefold    : termasuk terkait dengan transaksi yang kini makin banyak beralih melalui ruang digital.
tanpa stop  : termasuk terkait transaksi kini makin banyak beralih melalui ruang digital


## 6. Terapkan filter (kalimat < 5 kata, duplikat, kosong)

Satu-satunya pengurangan data yang dilakukan: buang kalimat terlalu pendek + duplikat + kosong. **Bukan** casefolding/stopword/filter topik.

In [ ]:
AMBANG_KATA = 5   # <- bisa diubah: ambang minimum kata per kalimat

kal_final = kal.copy()
kal_final = kal_final[kal_final["kalimat"].str.strip() != ""]          # buang kosong
kal_final = kal_final[kal_final["n_kata"] >= AMBANG_KATA]              # buang < 5 kata
kal_final = kal_final.drop_duplicates(subset="kalimat").reset_index(drop=True)  # buang duplikat
print(f"Kalimat sebelum filter : {len(kal)}")
print(f"Kalimat setelah filter : {len(kal_final)} (dibuang {len(kal)-len(kal_final)})")

## 7. Kesimpulan EDA

Rangkuman temuan dari analisis di atas (isi/ sesuaikan setelah melihat output di data penuh):

- **Jumlah data**: `N` artikel → `M` kalimat final (rata ± X kalimat/artikel).
- **Panjang kalimat**: mayoritas di rentang ... kata; sangat pendek (<5) dibuang karena sampah struktural (footer, penomoran, pecahan clickbait).
- **Duplikat**: ditemukan `d` kalimat identik (mis. boilerplate berulang) → dibuang.
- **Frekuensi kata**: didominasi stopword + kata domain finansial (rp, bank, saham) — wajar untuk berita bank; tidak ada kata yang dibuang berdasarkan frekuensi.
- **Stopword & casefolding**: hanya dianalisis, TIDAK diterapkan ke teks final (agar embedding BERTopic tetap utuh). Stopword akan ditangani di vectorizer Phase 03.
- **Keputusan**: data per kalimat disimpan apa adanya (teks asli), siap untuk BERTopic.